# ⚡ Async LangChain

## Why Async?
Synchronous code: one request at a time.
Async code: thousands of requests concurrently.

For production AI apps, async is **essential**.

## LangChain Async API
Every Runnable has async versions:
- `.ainvoke()` → async invoke
- `.astream()` → async streaming
- `.abatch()` → async batch (most efficient)

## FastAPI + LangChain
The most common production pattern:
FastAPI (async web framework) + LangChain (async chains)


In [ ]:
# ── Async Batch Processing ────────────────────────────────────────────────
import asyncio
import time
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

chain = (
    ChatPromptTemplate.from_template('Define {word} in 5 words.')
    | ChatOpenAI(model='gpt-4o-mini', temperature=0)
    | StrOutputParser()
)

words = ['Python', 'LangChain', 'embeddings', 'vector', 'agent', 'retriever']

async def process_async():
    start = time.time()
    # abatch runs ALL concurrently!
    results = await chain.abatch(
        [{'word': w} for w in words],
        config={'max_concurrency': 3}  # limit concurrent calls
    )
    elapsed = time.time() - start
    print(f'Async batch: {len(words)} calls in {elapsed:.2f}s')
    for word, result in zip(words, results):
        print(f'  {word}: {result}')

asyncio.run(process_async())

In [ ]:
# ── FastAPI Integration ────────────────────────────────────────────────
# This is the production pattern for LangChain web APIs.

fastapi_code = '''
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel

app = FastAPI()

chain = (
    ChatPromptTemplate.from_template("Answer: {question}")
    | ChatOpenAI(model="gpt-4o-mini")
    | StrOutputParser()
)

class QuestionRequest(BaseModel):
    question: str

@app.post("/chat")
async def chat(request: QuestionRequest):
    result = await chain.ainvoke({"question": request.question})
    return {"answer": result}

@app.post("/chat/stream")
async def chat_stream(request: QuestionRequest):
    async def generate():
        async for chunk in chain.astream({"question": request.question}):
            yield chunk
    return StreamingResponse(generate(), media_type="text/plain")
'''

print('FastAPI + LangChain pattern:')
print(fastapi_code)

## 🎯 Interview Questions

1. **Why use async LangChain in production?**
2. **What is abatch() and why is it more efficient?**
3. **How do you implement streaming in FastAPI with LangChain?**
4. **What is max_concurrency in abatch()?**

> Answer these before moving to the next module.

## 💪 Mini Exercise

**Exercise**: Based on what you learned in this module:

1. Re-run all code cells with your own inputs
2. Modify one code cell to add new functionality
3. Answer the interview questions above from memory

**Mini Project**: Build a small application that combines the concepts from this module.


## 📚 Summary

In this module you learned:
- The core concepts of **Async LangChain — Production Performance**
- How to implement them with modern LangChain/LangGraph APIs
- Common mistakes and how to avoid them
- Interview-level understanding of why each component exists

**Next**: Continue to the next module to build on these foundations.

---
*Part of the LangChain & LangGraph Master Course*
